# Week 2: From Experiments to Observational Data

This notebook holds the code from the Week 2 page, in the same order:
<https://yoonismo.github.io/causal-computing/week2.html>

Read the explanations on the page, run the cells here **in order** from top to bottom, and write your answers in the empty text cells.

## 0. Before you start

- Check **Select Kernel** at the top right: it should say **ECON 481 (Python 3.12)**.
- If a cell fails with `ModuleNotFoundError`, a package is missing from your environment. See `README.md`.
- The data sets are downloaded from the internet when you load them, so you need a connection.

## Lab 1 — A world where we know the truth

### 1.1 Create both potential outcomes

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy.stats import norm

import matplotlib.pyplot as plt

rng = np.random.default_rng(481)
n = 1000

y0 = rng.normal(loc=10, scale=3, size=n)       # Y(0): outcome without treatment
tau = 2                                        # the treatment effect, the same for everyone
y1 = y0 + tau                                  # Y(1): outcome with treatment

print("True ATE:", (y1 - y0).mean())

**Question 1.** Why can we compute the true ATE here, but never in real data?

*Your answer to Question 1:*

### 1.2 Randomize once

In [ ]:
d = rng.permutation(np.repeat([1, 0], n // 2))   # half treated, at random
y = d * y1 + (1 - d) * y0                         # what we would observe

print("Difference in means:", (y[d == 1].mean() - y[d == 0].mean()).round(3))

**Question 2.** The estimate is not exactly the true ATE. Is the estimator wrong? (The optional section at the end lets you check.)

*Your answer to Question 2:*

## Lab 2 — A real experiment: the National Supported Work Demonstration

Load the NSW data, then split it into the treated and the control group.

In [ ]:
from causaldata import nsw_mixtape

nsw = nsw_mixtape.load_pandas().data

In [ ]:
treated = nsw[nsw["treat"] == 1]
control = nsw[nsw["treat"] == 0]

Group sizes: $n_1$ treated, $n_0$ control, $n = n_1 + n_0$. The whole sample size is called `n_nsw`, because `n` already holds the size of the simulated world in Lab 1.

In [ ]:
n_nsw = nsw.shape[0] ## the entire sample size
n_1 = len(nsw[nsw["treat"]==1])## the number of treated 
n_0 = len(nsw[nsw["treat"]==0]) ## the number of control units

print("the number of treated: ", n_1, "the number of control: ", n_0)

Group means $\overline{Y}_1$ and $\overline{Y}_0$: the sum of the outcomes in each group divided by the number of people in it.

In [ ]:
ybar_1 = treated["re78"].sum()/n_1   # mean outcome of the treated group
ybar_0 = control["re78"].sum()/n_0   # mean outcome of the control group

diff = ybar_1 - ybar_0

Sample variances $\hat{S}_1^2$ and $\hat{S}_0^2$, and the standard error $\text{SE} = \sqrt{\hat{S}_1^2 / n_1 + \hat{S}_0^2 / n_0}$.

In [ ]:
s2_1 = np.sum((treated["re78"] - ybar_1)**2)/(n_1-1)   # sample variance in the treated group
s2_0 = np.sum((control["re78"] - ybar_0)**2)/(n_0-1)   # sample variance in the control group

se = np.sqrt(s2_1/n_1 + s2_0/n_0)

The 95% confidence interval.

In [ ]:
quantile_975 = norm.ppf(0.975)
print("95% interval: ", (diff -  quantile_975* se).round(1), "to", (diff + quantile_975 * se).round(1))

The same steps in a few lines of `pandas`:

In [ ]:
from causaldata import nsw_mixtape

nsw = nsw_mixtape.load_pandas().data
treated = nsw[nsw["treat"] == 1]
control = nsw[nsw["treat"] == 0]

diff = treated["re78"].mean() - control["re78"].mean()
se = np.sqrt(treated["re78"].var() / len(treated) + control["re78"].var() / len(control))

print("Difference in means:", diff.round(1))
print("Standard error:     ", se.round(1))
print("95% interval:       ", (diff - 1.96 * se).round(1), "to", (diff + 1.96 * se).round(1))

### The same numbers in one line

A regression of `re78` on `treat` with `cov_type="HC2"` gives the same estimate and standard error.

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols("re78 ~ treat", data=nsw).fit(cov_type="HC2")

print("Difference in means:", model.params["treat"].round(1))
print("Standard error:     ", model.bse["treat"].round(1))
print("95% interval:       ", model.conf_int().loc["treat"].round(1).tolist())

### Balance check

In [ ]:
covariates = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]

balance = pd.DataFrame({
    "treated": treated[covariates].mean(),
    "control": control[covariates].mean(),
})
balance["std. diff."] = (balance["treated"] - balance["control"]) / np.sqrt(
    (treated[covariates].var() + control[covariates].var()) / 2
)
print(balance.round(2))

**Question 3.** Most standardized differences are small, but `nodegree` is about −0.30. Does that mean the randomization failed?

*Your answer to Question 3:*

## Lab 3 — Observational studies

### 3.2 A simulation: let people choose

In [ ]:
p = 1 / (1 + np.exp((y0 - 10) / 2))     # lower y0 -> higher chance of treatment
d_sel = rng.binomial(1, p)
y_sel = d_sel * y1 + (1 - d_sel) * y0

naive = y_sel[d_sel == 1].mean() - y_sel[d_sel == 0].mean()
att = (y1 - y0)[d_sel == 1].mean()
selection_bias = y0[d_sel == 1].mean() - y0[d_sel == 0].mean()

print("Naive difference:", naive.round(3))
print("ATT:             ", att.round(3))
print("Selection bias:  ", selection_bias.round(3))
print("ATT + bias:      ", (att + selection_bias).round(3))

**Question 4.** The naive difference is negative although the effect is exactly 2 for everyone. Which line explains why? Check that the last two lines add up.

*Your answer to Question 4:*

### 3.3 LaLonde (1986): swap the control group

In [ ]:
from causaldata import cps_mixtape

cps = cps_mixtape.load_pandas().data
print("Treated minus CPS:", (treated["re78"].mean() - cps["re78"].mean()).round(1))

In [ ]:
pd.DataFrame({
    "NSW treated": treated[covariates].mean(),
    "NSW control": control[covariates].mean(),
    "CPS": cps[covariates].mean(),
}).round(2)

**Question 5.** Taking 1,794 as the ATT, how large is the selection bias? Which rows of the table explain its sign?

*Your answer to Question 5:*

### 3.4 What if we keep only CPS men who look like the trainees?

In [ ]:
groups = {
    "all CPS": cps,
    "CPS, re75 below 5,000": cps[cps["re75"] < 5000],
    "CPS, no earnings in 1974 and 1975": cps[(cps["re74"] == 0) & (cps["re75"] == 0)],
}
for label, comparison in groups.items():
    gap = treated["re78"].mean() - comparison["re78"].mean()
    print(f"{label:35s} n = {len(comparison):6d}   gap = {gap:9.1f}")

**Question 6.** The answer moves a lot depending on which rule you pick. What would a *principled* way of choosing comparison people need to do?

*Your answer to Question 6:*

## Optional: randomize many times

In [ ]:
reps = 2000
D = rng.permuted(np.tile(np.repeat([1, 0], n // 2), (reps, 1)), axis=1)
Y = D * y1 + (1 - D) * y0

estimates = (Y * D).sum(axis=1) / D.sum(axis=1) - (Y * (1 - D)).sum(axis=1) / (1 - D).sum(axis=1)
print("Estimated ATE:", (np.sum(estimates)/reps).round(3))
plt.hist(estimates, bins=40, color="lightblue", edgecolor="white")
plt.axvline(tau, color="red", label="true ATE")
plt.axvline(np.sum(estimates)/reps, color="blue", linestyle="--",label="estimated ATE")
plt.legend()
plt.show()

**Optional question.** Where is the centre of the histogram? What does that tell you about the difference in means under random assignment?

*Your answer to the optional question:*

## Next

Open `star_practice.ipynb` and repeat Labs 2 and 3 on your own with the Tennessee STAR experiment.